# Gaia BH1 — Stellar Wind from G-Type Companion onto a Dormant 9.6 M$_\odot$ Black Hole

Test notebook for the SERPENS+GR pipeline applied to Gaia BH1 (El-Badry et al. 2023).
We launch a thermally-driven stellar wind from the G-type companion, follow particles under 1PN gravity, and predict observables: BHL capture rate, sky-projected column density at the binary inclination, and Doppler line profiles.

In [ ]:
import os
WORKDIR = '/Users/raghavchari/SERPENS'
if os.getcwd() != WORKDIR:
    os.chdir(WORKDIR)
print('Working Directory:', os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.serpens_simulation import SerpensSimulation
from src.parameters import GLOBAL_PARAMETERS
from src.species import Species
from src import bh_observables as bho

## 1. Configure simulation: GR enabled, wind parameters, build system

In [ ]:
# GR
GLOBAL_PARAMETERS.set('gr_enabled', True)
GLOBAL_PARAMETERS.set('gr_source', 'bh')

# Stellar wind launch parameters (typical solar/G-dwarf values)
GLOBAL_PARAMETERS.set('wind_T_corona', 1.5e6)   # K
GLOBAL_PARAMETERS.set('wind_v_inf', 4.0e5)      # m/s (asymptotic v ~ 400 km/s)

# Build Gaia BH1 system
sim = SerpensSimulation(system='GaiaBH1')

bh = sim.particles['bh']
comp = sim.particles['companion']
print(f'BH mass: {bh.m:.3e} kg ({bh.m/1.989e30:.2f} M_sun)')
print(f'Companion mass: {comp.m:.3e} kg ({comp.m/1.989e30:.2f} M_sun)')
print(f'Schwarzschild radius: {sim._r_schwarzschild:.3e} m')
print(f'ISCO: {sim._r_isco:.3e} m')

orbital_period_s = comp.orbit(primary=bh).P
print(f'Orbital period: {orbital_period_s/86400:.1f} days')

## 2. Define wind species

We launch H, He, C, O — the dominant solar-wind species — with relative number abundances roughly tracking solar composition. `n_wind` is the number of superparticles spawned per launch event per species.

In [ ]:
# Solar-wind-like mass loss rate (G-dwarf): ~2e-14 Msun/yr ~ 1.3e9 kg/s
Mdot_total = 1.3e9   # kg/s

# Per-species mass-loss fractions (rough solar abundance by mass)
frac = {'H': 0.74, 'He': 0.245, 'O': 0.01, 'C': 0.005}
n_per_species = 30   # superparticles per spawn

wind_species = []
for name, f in frac.items():
    s = Species(
        name=name,
        n_th=0, n_sp=0,
        n_wind=n_per_species,
        mass_per_sec=Mdot_total * f,
        beta=0.0,
        lifetime=1e10
    )
    wind_species.append(s)
    print(s)

In [ ]:
sim.object_to_source('companion', wind_species)

## 3. Run simulation

Integrate for two orbital periods with 30 spawn events, giving us a phase-resolved snapshot ensemble.

In [ ]:
sim.advance(orbits=2, spawns=30, orbits_reference='companion', verbose=False)
print(f'Final particle count: {sim.N - sim.N_active}')

## 4. Extract particle data and BHL capture diagnostic

In [ ]:
bh = sim.particles['bh']
comp = sim.particles['companion']
r_s = sim._r_schwarzschild

n_test = sim.N - sim.N_active
pos = np.array([[sim.particles[i].x, sim.particles[i].y, sim.particles[i].z]
                for i in range(sim.N_active, sim.N)])
vel = np.array([[sim.particles[i].vx, sim.particles[i].vy, sim.particles[i].vz]
                for i in range(sim.N_active, sim.N)])

bh_pos = np.array([bh.x, bh.y, bh.z])
bh_vel = np.array([bh.vx, bh.vy, bh.vz])
comp_pos = np.array([comp.x, comp.y, comp.z])

# BHL capture: which particles lie within r_BHL of the BH at this snapshot?
captured = bho.captured_particle_mask(pos, vel, bh_pos, bh_vel, bh.m, factor=1.0)
print(f'Particles inside Bondi-Hoyle radius: {captured.sum()} / {n_test}'
      f' ({100*captured.mean():.2f}%)')

# Analytic BHL capture rate using mean v_rel and a back-of-envelope wind density
v_rel_mean = np.linalg.norm(vel - bh_vel, axis=1).mean()
binary_sep = np.linalg.norm(comp_pos - bh_pos)
rho_wind_estimate = (1.3e9) / (4 * np.pi * binary_sep**2 * 4.0e5)   # Mdot/(4 pi r^2 v)
Mdot_bhl = bho.bhl_mass_capture_rate(bh.m, rho_wind_estimate, v_rel_mean)
print(f'Analytical BHL Mdot estimate: {Mdot_bhl:.2e} kg/s'
      f' = {Mdot_bhl/1.989e30 * 365.25*86400:.2e} M_sun/yr')

## 5. Planar view of the binary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10), facecolor='black')
ax.set_facecolor('black')
for s in ax.spines.values():
    s.set_color('white')
ax.tick_params(colors='white')

AU = 1.496e11
x_rel = (pos[:, 0] - bh.x) / AU
y_rel = (pos[:, 1] - bh.y) / AU
speeds = np.linalg.norm(vel - bh_vel, axis=1) / 1e3

sc = ax.scatter(x_rel, y_rel, c=speeds, s=2, alpha=0.6, cmap='plasma')
ax.scatter(0, 0, c='white', s=80, marker='*', zorder=10, label='BH')
ax.scatter((comp.x - bh.x)/AU, (comp.y - bh.y)/AU, c='gold', s=120, marker='o',
           edgecolors='white', linewidths=1, zorder=10, label='G-dwarf companion')

# Highlight BHL-captured particles
if captured.any():
    ax.scatter(x_rel[captured], y_rel[captured], facecolors='none',
               edgecolors='red', s=20, label=f'within r_BHL ({captured.sum()})')

ax.set_aspect('equal')
ax.set_xlabel('x [AU]', color='white', fontsize=14)
ax.set_ylabel('y [AU]', color='white', fontsize=14)
ax.set_title(f'Gaia BH1 — wind from G-dwarf at {n_test} particles', color='white', fontsize=14)
cb = plt.colorbar(sc, ax=ax)
cb.set_label('|v - v_BH| [km/s]', color='white')
cb.ax.tick_params(colors='white')
ax.legend(facecolor='black', edgecolor='white', labelcolor='white', loc='upper right')
plt.tight_layout()
plt.show()

## 6. Observer-frame projection (binary inclination = 126.6°)

In [ ]:
inclination = np.deg2rad(126.6)   # El-Badry+2023

# Center on the BH then project
pos_centered = pos - bh_pos
vel_centered = vel - bh_vel
sky_pos, los_vel = bho.project_to_observer(pos_centered, vel_centered, inclination)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='white')

# Sky projection
ax = axes[0]
ax.scatter(sky_pos[:, 0]/AU, sky_pos[:, 1]/AU, c=los_vel/1e3, s=2, alpha=0.6, cmap='RdBu_r',
           vmin=-300, vmax=300)
comp_sky, _ = bho.project_to_observer((comp_pos - bh_pos)[None, :], bh_vel[None, :] * 0,
                                       inclination)
ax.scatter(0, 0, c='black', s=80, marker='*', label='BH')
ax.scatter(comp_sky[0, 0]/AU, comp_sky[0, 1]/AU, c='gold', s=120, marker='o',
           edgecolors='black', linewidths=1, label='G-dwarf')
ax.set_aspect('equal')
ax.set_xlabel('X_sky [AU]', fontsize=12)
ax.set_ylabel('Y_sky [AU]', fontsize=12)
ax.set_title(f'Sky projection at inclination {np.rad2deg(inclination):.1f}°', fontsize=13)
ax.legend()

# Column density along LOS
ax = axes[1]
extent_au = np.max(np.abs(sky_pos[:, :2]/AU)) * 1.05
edges = np.linspace(-extent_au, extent_au, 80)
H, xe, ye = np.histogram2d(sky_pos[:, 0]/AU, sky_pos[:, 1]/AU, bins=[edges, edges])
im = ax.imshow(np.log10(H.T + 0.5), origin='lower',
               extent=[xe[0], xe[-1], ye[0], ye[-1]], cmap='inferno', aspect='equal')
ax.set_xlabel('X_sky [AU]', fontsize=12)
ax.set_ylabel('Y_sky [AU]', fontsize=12)
ax.set_title('Sky column density (log)', fontsize=13)
plt.colorbar(im, ax=ax, label='log$_{10}$ count')

plt.tight_layout()
plt.show()

## 7. Doppler line profile (rest-frame Lyman-α)

In [ ]:
# Filter to H particles only — match species id 5
species_ids = np.array([sim.get_particle_param(sim.particles[i].hash.value, 'serpens_species')
                        for i in range(sim.N_active, sim.N)])
h_mask = species_ids == 5
print(f'Hydrogen test particles: {h_mask.sum()}')

if h_mask.any():
    v_centers, flux, wavelengths = bho.doppler_line_profile(
        los_vel[h_mask],
        rest_wavelength=1215.67,        # Lyman-alpha [Angstrom]
        v_min=-1000e3, v_max=1000e3,
        n_bins=200,
        thermal_broadening=20e3         # ~20 km/s thermal width
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(v_centers/1e3, flux, c='steelblue')
    axes[0].axvline(0, color='gray', linestyle=':', alpha=0.6)
    axes[0].set_xlabel('LOS velocity [km/s]')
    axes[0].set_ylabel('column density (arb. units)')
    axes[0].set_title('Doppler line profile — H Ly$\\alpha$')

    axes[1].plot(wavelengths, flux, c='steelblue')
    axes[1].axvline(1215.67, color='gray', linestyle=':', alpha=0.6, label='rest 1215.67 Å')
    axes[1].set_xlabel(r'Wavelength [\AA]')
    axes[1].set_ylabel('column density (arb. units)')
    axes[1].set_title(r'H Ly$\alpha$ — observer wavelength frame')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print('No H particles in snapshot.')